# 🧬 PrismBB Drug — One-Click Cloud Demo

**AI-Powered Drug Discovery Platform** — runs entirely in your browser using a free Google Colab machine. No installation required on your computer.

**What you get:**
- 🔬 SMILES → 17+ molecular descriptors + 3D conformer
- 💊 104 ADMET predictions (absorption, distribution, metabolism, excretion, toxicity)
- ⚛️ AutoDock Vina protein-ligand docking with grid box + scoring functions
- 🎨 Modern web UI with interactive 3D molecular viewer

---

## 👉 How to use this notebook (no coding required)

1. **(Recommended) Switch to GPU** for faster ADMET predictions:  
   `Runtime` → `Change runtime type` → `Hardware accelerator` → **T4 GPU** → Save
2. Click `Runtime` → **`Run all`** (or press `Ctrl+F9` / `⌘F9`)
3. Wait ~3 minutes while everything sets up
4. A big **"Open the app"** button will appear at the bottom — click it
5. Keep this Colab tab open while you use the app

When you're done, close the Colab tab to stop the server.

## Step 1 · Check the runtime

In [ ]:
import subprocess, sys

print(f'🐍 Python: {sys.version.split()[0]}')

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True)
if gpu.returncode == 0 and gpu.stdout.strip():
    print(f'✅ GPU: {gpu.stdout.strip()}')
    print('   → ADMET predictions will run on GPU (fast).')
else:
    print('⚠️  No GPU enabled.')
    print('   → ADMET predictions will run on CPU (slower but still works).')
    print('   → To enable: Runtime → Change runtime type → T4 GPU → Save → restart cell.')

## Step 2 · Install dependencies

Installs AutoDock Vina (the real docking engine), RDKit, ADMET-AI, and the rest. Takes ~2 minutes.

In [ ]:
print('📦  Installing AutoDock Vina + system libs ...')
!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq autodock-vina libxrender1 libxext6 libsm6 > /dev/null

print('📦  Installing Python packages (rdkit, biopython, meeko, agno, flask) ...')
!pip install -q \
    'fastapi>=0.112' 'uvicorn[standard]>=0.30' pydantic python-multipart \
    'rdkit>=2024.3.1' 'biopython>=1.83' 'meeko>=0.7.0' 'gemmi>=0.6.5' \
    'agno>=2.0,<3' \
    flask requests python-dotenv

print('📦  Installing ADMET-AI (this pulls torch ~2 GB if not already present) ...')
!pip install -q 'admet-ai>=1.4.0'

print('✅  All dependencies installed.')
import shutil
print(f'   • Vina binary: {shutil.which("vina") or "NOT FOUND"}')

## Step 3 · Get the PrismBB Drug source

Clones the latest version from GitHub.

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/Babajan-B/PrismBB-Drug.git'
REPO_DIR = '/content/PrismBB-Drug'

if os.path.exists(REPO_DIR):
    print('↻  Updating existing checkout ...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)
else:
    print(f'📥  Cloning {REPO_URL} ...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print(f'📁  Working in: {os.getcwd()}')
print('   • backend/  :', os.path.isdir('backend'))
print('   • frontend/ :', os.path.isdir('flask_frontend'))

## Step 4 · Launch the backend (FastAPI on port 8000)

Starts the multi-agent backend in the background and waits for it to become healthy.

In [ ]:
import os, sys, time, subprocess, urllib.request

# Make sure we're in the repo root
os.chdir('/content/PrismBB-Drug')

backend_log = open('/content/backend.log', 'w')
backend_env = {
    **os.environ,
    'PYTHONPATH': '/content/PrismBB-Drug/backend',
    'AGNO_TELEMETRY': 'false',
    'ADMET_AI_CACHE_DIR': '/content/admet_cache',
    'HF_HOME': '/content/hf_cache',
}
backend_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app.main:app',
     '--host', '0.0.0.0', '--port', '8000', '--log-level', 'warning'],
    cwd='backend',
    env=backend_env,
    stdout=backend_log, stderr=subprocess.STDOUT,
)

print('🚀  Backend starting on http://localhost:8000 ...')
for i in range(40):
    try:
        urllib.request.urlopen('http://localhost:8000/api/health', timeout=2)
        print('✅  Backend is healthy')
        break
    except Exception:
        time.sleep(1)
else:
    print('❌  Backend failed to start — last 50 lines of log:')
    backend_log.flush()
    with open('/content/backend.log') as f:
        print(''.join(f.readlines()[-50:]))
    raise RuntimeError('Backend startup failed')

## Step 5 · Launch the Flask web UI (port 3000)

The UI is what end users actually see. It proxies API calls to the FastAPI backend internally.

In [ ]:
import os, sys, time, subprocess, urllib.request

frontend_log = open('/content/frontend.log', 'w')
frontend_env = {
    **os.environ,
    'BACKEND_URL': 'http://localhost:8000',
    'PORT': '3000',
    'FLASK_DEBUG': '0',
}
frontend_proc = subprocess.Popen(
    [sys.executable, 'app.py'],
    cwd='/content/PrismBB-Drug/flask_frontend',
    env=frontend_env,
    stdout=frontend_log, stderr=subprocess.STDOUT,
)

print('🚀  Web UI starting on http://localhost:3000 ...')
for i in range(30):
    try:
        urllib.request.urlopen('http://localhost:3000/', timeout=2)
        print('✅  Web UI is ready')
        break
    except Exception:
        time.sleep(1)
else:
    print('❌  Frontend failed to start — last 30 lines of log:')
    frontend_log.flush()
    with open('/content/frontend.log') as f:
        print(''.join(f.readlines()[-30:]))
    raise RuntimeError('Frontend startup failed')

## Step 6 · Expose a public URL via Cloudflare Tunnel

Creates a temporary public URL like `https://something-random.trycloudflare.com` that **anyone can open in their browser**. No signup, no account needed. The URL lives until this notebook stops.

In [ ]:
import os, re, subprocess, threading, time
from IPython.display import HTML, display

# Download cloudflared (single static binary, ~30 MB)
if not os.path.exists('/content/cloudflared'):
    print('⬇️   Downloading cloudflared ...')
    subprocess.run([
        'wget', '-q', '-O', '/content/cloudflared',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
    ], check=True)
    os.chmod('/content/cloudflared', 0o755)

tunnel_log = open('/content/tunnel.log', 'w')
tunnel = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://localhost:3000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

public_url = None
def watch_tunnel():
    global public_url
    for raw in tunnel.stdout:
        line = raw.decode('utf-8', errors='ignore')
        tunnel_log.write(line)
        tunnel_log.flush()
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m and not public_url:
            public_url = m.group(0)

threading.Thread(target=watch_tunnel, daemon=True).start()

print('🌐  Waiting for public URL ...')
for _ in range(45):
    if public_url:
        break
    time.sleep(1)

if public_url:
    display(HTML(f"""
    <div style='padding:28px; background:linear-gradient(135deg,#0b0d12,#11141c); border:1px solid #2a2f3d; border-radius:18px; text-align:center; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; box-shadow: 0 12px 40px rgba(124,92,255,.18);'>
        <div style='font-size:38px; line-height:1'>🧬</div>
        <h2 style='color:#e6e9f2; margin:14px 0 6px; font-weight:700; letter-spacing:-.01em'>PrismBB Drug is live!</h2>
        <p style='color:#9aa3b8; margin:0 0 20px'>Click the button below to open the app in a new tab.</p>
        <a href='{public_url}' target='_blank' style='display:inline-block; padding:14px 32px; background:linear-gradient(135deg,#7c5cff 0%,#5d8bff 50%,#22d3ee 100%); color:white; text-decoration:none; border-radius:12px; font-weight:600; font-size:1.05rem; box-shadow:0 10px 30px rgba(124,92,255,.35)'>
            👉 Open the app
        </a>
        <p style='color:#6b7388; margin:18px 0 0; font-family: "JetBrains Mono", monospace; font-size:.85rem; word-break:break-all'>{public_url}</p>
        <div style='margin-top:18px; padding-top:18px; border-top:1px solid #2a2f3d; color:#9aa3b8; font-size:.85rem'>
            ✅ Keep this Colab tab open · Backend on GPU · Real AutoDock Vina docking enabled
        </div>
    </div>
    """))
else:
    print('❌  Tunnel failed to provide a URL. Last 20 lines of tunnel.log:')
    with open('/content/tunnel.log') as f:
        print(''.join(f.readlines()[-20:]))

## 🟢 You're live!

**Click the big purple button above** to open PrismBB Drug. You'll see:

| Page | What it does |
|---|---|
| **Analyze** | Paste any SMILES (e.g. `CC(=O)Oc1ccccc1C(=O)O` for aspirin) → 17 descriptors + 3D structure + 104 ADMET predictions in a searchable table |
| **Docking** | Upload a protein PDB + ligand PDB/SDF → real AutoDock Vina docking with poses, affinities, and 3D pose viewer |
| **Examples** | Curated demo molecules (aspirin, caffeine, ibuprofen, …) |
| **About** | Architecture overview |

### Troubleshooting

- **"App took too long to respond"** — Hit refresh; the first ADMET call downloads ~2 GB of model weights (one-time, takes a minute).
- **"This site can't be reached"** — Colab disconnected (90 min idle on free tier). Re-run all cells.
- **Want a permanent URL?** — Deploy to Hugging Face Spaces or Render; see `DEPLOY.md` in the repo.

### When you're done

Just close this Colab tab. The tunnel stops automatically and your free runtime is released.

---
## (Optional) Keep server alive · stream logs · stop manually

In [ ]:
# Watch live backend logs (interrupt the cell with the ⏹ button to stop watching).
# !tail -f /content/backend.log

# Watch live frontend logs:
# !tail -f /content/frontend.log

# Stop everything manually:
# backend_proc.terminate(); frontend_proc.terminate(); tunnel.terminate()

print('Tip: uncomment a line above and run this cell to inspect logs or stop services.')